Importuojamos bibliotekos.


In [ ]:
import logging
import time
import warnings
from itertools import product
from pathlib import Path

import numpy as np
import optuna
import pandas as pd

from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MAE
from neuralforecast.models import LSTM

warnings.filterwarnings("ignore")

Nustatomi failų keliai ir pagrindiniai modeliavimo parametrai.


In [ ]:
DATA_DIR = Path(r"...")
OUTPUT_DIR = DATA_DIR / "lstm_results_3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEST_PREDICTIONS_46_FILE = OUTPUT_DIR / "lstm_test_predictions_46.parquet"

DATA_FILE = DATA_DIR / "main_lentele_50_su_features.parquet"
TEST_SET_FILE = DATA_DIR / "test_set_46.parquet"

H = 7
N_WINDOWS = 26
VAL_DAYS = H * N_WINDOWS
TEST_DAYS = H * N_WINDOWS
EXPECTED_FINAL_46_TEST_ROWS = 46 * N_WINDOWS * H

RANDOM_SEED = 67
RUN_FULL_GRID = False
RUN_TEST = False
OPTUNA_N_TRIALS = 60

main = pd.read_parquet(DATA_FILE)
selected_test = pd.read_parquet(TEST_SET_FILE)

Paruošiami identifikatoriai, datos, tikslinis kintamasis ir metaduomenys.


In [ ]:
ID_COMPONENT_COLS = ["STORE_ID", "SKU_GROUP", "SKU_ID"]
TARGET_COL = "SALES_QTY"
DATE_COL = "DATE"

REPORTING_META_COLS = [
    "unique_id",
    "STORE_ID",
    "SKU_GROUP",
    "SKU_ID",
    "sales_level",
    "zero_level",
    "series_type_2d",
    "mean_sales",
    "zero_pct",
]

FORBIDDEN_MODEL_FEATURES = {
    "sales_level",
    "zero_level",
    "series_type_2d",
    "mean_sales",
    "zero_pct",
    "STORE_ID",
    "SKU_GROUP",
    "SKU_ID",
    "unique_id",
    "DATE",
    "SALES_QTY",
    "DAY_OF_WEEK",
}


def ensure_unique_id(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "unique_id" not in df.columns:
        missing_id_cols = [col for col in ID_COMPONENT_COLS if col not in df.columns]
        if missing_id_cols:
            raise ValueError(f"Cannot create unique_id; missing columns: {missing_id_cols}")
        df["unique_id"] = df[ID_COMPONENT_COLS].astype(str).agg("_".join, axis=1)
    else:
        df["unique_id"] = df["unique_id"].astype(str)
    return df


main = ensure_unique_id(main)
selected_test = ensure_unique_id(selected_test)

main[DATE_COL] = pd.to_datetime(main[DATE_COL])
selected_test[DATE_COL] = pd.to_datetime(selected_test[DATE_COL])

main["ds"] = main[DATE_COL]
main["y"] = pd.to_numeric(main[TARGET_COL], errors="coerce").astype("float32")

selected_46_ids = sorted(selected_test["unique_id"].dropna().astype(str).unique())
selected_meta_cols = [col for col in REPORTING_META_COLS if col in selected_test.columns]
selected_metadata = (
    selected_test[selected_meta_cols]
    .drop_duplicates(subset=["unique_id"])
    .reset_index(drop=True)
)
selected_metadata = selected_metadata[selected_metadata["unique_id"].isin(selected_46_ids)].copy()


Paruošiami išoriniai kintamieji ir duomenų rinkinys dienos lygyje.


In [ ]:
POTENTIAL_FUTR_EXOG_COLS = [
    "PROMO_FLAG",
    "HOLIDAY_FLAG",
    "AVG_TEMP",
    "AVG_FEEL_TEMP",
    "TOTAL_PRECIP",
    "AVG_WIND",
    "YESTERDAY_HOLIDAY_FLAG",
    "TOMORROW_HOLIDAY_FLAG",
    "WEEKEND_FLAG",
]

BINARY_FUTR_EXOG_COLS = [
    "PROMO_FLAG",
    "HOLIDAY_FLAG",
    "YESTERDAY_HOLIDAY_FLAG",
    "TOMORROW_HOLIDAY_FLAG",
    "WEEKEND_FLAG",
]

CONTINUOUS_FUTR_EXOG_COLS = [
    "AVG_TEMP",
    "AVG_FEEL_TEMP",
    "TOTAL_PRECIP",
    "AVG_WIND",
]

available_base_exog_cols = [col for col in POTENTIAL_FUTR_EXOG_COLS if col in main.columns]
main_meta_cols = [col for col in ID_COMPONENT_COLS if col in main.columns]
main_daily_cols = ["unique_id", "ds", "y"] + available_base_exog_cols + main_meta_cols
main_daily_cols = list(dict.fromkeys(main_daily_cols))

agg_spec = {"y": "sum"}
for col in available_base_exog_cols + main_meta_cols:
    if col not in agg_spec:
        agg_spec[col] = "first"

main_daily = (
    main[main_daily_cols]
    .groupby(["unique_id", "ds"], as_index=False)
    .agg(agg_spec)
)

all_ids = sorted(main_daily["unique_id"].unique())
all_dates = pd.date_range(main_daily["ds"].min(), main_daily["ds"].max(), freq="D")
full_index = pd.MultiIndex.from_product([all_ids, all_dates], names=["unique_id", "ds"])

main_full = (
    main_daily
    .set_index(["unique_id", "ds"])
    .reindex(full_index)
    .reset_index()
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

for col in main_meta_cols:
    main_full[col] = main_full.groupby("unique_id", sort=False)[col].transform(lambda s: s.ffill().bfill())

main_full["y"] = pd.to_numeric(main_full["y"], errors="coerce").fillna(0).astype("float32")

for col in [c for c in BINARY_FUTR_EXOG_COLS if c in main_full.columns and c != "WEEKEND_FLAG"]:
    main_full[col] = pd.to_numeric(main_full[col], errors="coerce").fillna(0).astype("float32")

for col in [c for c in CONTINUOUS_FUTR_EXOG_COLS if c in main_full.columns]:
    main_full[col] = pd.to_numeric(main_full[col], errors="coerce")
    main_full[col] = main_full.groupby("unique_id", sort=False)[col].ffill()
    main_full[col] = main_full[col].fillna(0).astype("float32")

main_full["DAY_OF_WEEK"] = main_full["ds"].dt.dayofweek.astype("int8")
if "WEEKEND_FLAG" in available_base_exog_cols:
    main_full["WEEKEND_FLAG"] = (main_full["DAY_OF_WEEK"] >= 5).astype("float32")


Užkoduojama savaitės diena ir atrenkami statiniai požymiai.


In [ ]:
DOW_COLS = [f"DOW_{i}" for i in range(1, 7)]

dow_dummies = pd.get_dummies(
    pd.Categorical(main_full["DAY_OF_WEEK"], categories=list(range(7))),
    prefix="DOW",
    drop_first=True,
).astype("float32")

for col in DOW_COLS:
    if col not in dow_dummies.columns:
        dow_dummies[col] = np.float32(0)

dow_dummies = dow_dummies[DOW_COLS]
main_full = pd.concat([main_full.drop(columns=DOW_COLS, errors="ignore"), dow_dummies], axis=1)

futr_exog_cols = [col for col in POTENTIAL_FUTR_EXOG_COLS if col in main_full.columns] + DOW_COLS
futr_exog_cols = list(dict.fromkeys(futr_exog_cols))

bad_futr_cols = sorted(set(futr_exog_cols) & FORBIDDEN_MODEL_FEATURES)
if bad_futr_cols:
    raise ValueError(f"Forbidden metadata/raw columns leaked into futr_exog_cols: {bad_futr_cols}")

if "DAY_OF_WEEK" in futr_exog_cols:
    raise ValueError("Raw DAY_OF_WEEK must not be included in futr_exog_cols.")

for col in futr_exog_cols:
    main_full[col] = pd.to_numeric(main_full[col], errors="coerce").fillna(0).astype("float32")


Suformuojamas duomenų rinkinys.


In [ ]:
Y_df_full = main_full[["unique_id", "ds", "y"] + futr_exog_cols].copy()
Y_df_full = Y_df_full.sort_values(["unique_id", "ds"]).reset_index(drop=True)
Y_df_full["y"] = Y_df_full["y"].astype("float32")


Apibrėžiamos validacijos ir testavimo datos.


In [ ]:
max_date = Y_df_full["ds"].max()
test_start = max_date - pd.Timedelta(days=TEST_DAYS - 1)
val_start = test_start - pd.Timedelta(days=VAL_DAYS)
val_end = test_start - pd.Timedelta(days=1)
train_val_df = Y_df_full[Y_df_full["ds"] < test_start].copy()

Apibrėžiama WAPE metrika modelio parinkimui.


In [ ]:
PRED_COL = "LSTM"


def wape(actual, predicted) -> float:
    y_true = pd.to_numeric(pd.Series(actual), errors="coerce").to_numpy(dtype=float)
    y_pred = pd.to_numeric(pd.Series(predicted), errors="coerce").to_numpy(dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        return np.nan
    return float(np.sum(np.abs(y_true - y_pred)) / denom)


def forecast_wape(df: pd.DataFrame, pred_col: str = PRED_COL) -> float:
    return wape(df["y"], df[pred_col])


Sudaroma LSTM hiperparametrų paieškos gardelė.


In [ ]:
param_grid = {
    "input_size": [21, 28, 35, 42],
    "hidden_size": [32, 64],
    "n_layers": [1, 2],
    "learning_rate": [0.01, 0.001, 0.0005],
    "batch_size": [64, 128],
    "max_steps": [1000, 3000, 5000],
    "random_seed": [RANDOM_SEED],
    "scaler_type": ["standard"],
}

test_param_grid = {
    "input_size": [28],
    "hidden_size": [32],
    "n_layers": [1],
    "learning_rate": [0.001],
    "batch_size": [128],
    "max_steps": [1000],
    "random_seed": [RANDOM_SEED],
    "scaler_type": ["standard"],
}


def expand_grid(grid: dict) -> list[dict]:
    keys = list(grid.keys())
    return [dict(zip(keys, values)) for values in product(*(grid[key] for key in keys))]


def tie_lstm_params(params: dict) -> dict:
    params = params.copy()
    hidden_size = int(params.pop("hidden_size"))
    n_layers = int(params.pop("n_layers"))

    params["encoder_hidden_size"] = hidden_size
    params["decoder_hidden_size"] = hidden_size
    params["encoder_n_layers"] = n_layers
    params["decoder_layers"] = n_layers
    return params


full_grid = [tie_lstm_params(params) for params in expand_grid(param_grid)]
test_grid = [tie_lstm_params(params) for params in expand_grid(test_param_grid)]

for params in full_grid + test_grid:
    if params["encoder_hidden_size"] != params["decoder_hidden_size"]:
        raise ValueError("encoder_hidden_size and decoder_hidden_size must be tied.")
    if params["encoder_n_layers"] != params["decoder_layers"]:
        raise ValueError("encoder_n_layers and decoder_layers must be tied.")

if RUN_TEST:
    optuna_search_grid = test_param_grid
    optuna_sampler_name = "GridSampler"
    optuna_n_trials = len(test_grid)
elif RUN_FULL_GRID:
    optuna_search_grid = param_grid
    optuna_sampler_name = "GridSampler"
    optuna_n_trials = len(full_grid)
else:
    optuna_search_grid = param_grid
    optuna_sampler_name = "TPESampler"
    optuna_n_trials = min(int(OPTUNA_N_TRIALS), len(full_grid))

Vykdoma Optuna validacija pagal WAPE.


In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
if optuna_sampler_name == "GridSampler":
    optuna_sampler = optuna.samplers.GridSampler(optuna_search_grid, seed=RANDOM_SEED)
elif optuna_sampler_name == "TPESampler":
    optuna_sampler = optuna.samplers.TPESampler(
        seed=RANDOM_SEED,
        n_startup_trials=min(10, optuna_n_trials),
    )
else:
    raise ValueError(f"Unsupported Optuna sampler: {optuna_sampler_name}")

study = optuna.create_study(
    direction="minimize",
    sampler=optuna_sampler,
    study_name=f"lstm_neuralforecast_{optuna_sampler_name.lower()}",
)

validation_results = []

def suggest_lstm_params(trial: optuna.Trial) -> dict:
    raw_params = {
        name: trial.suggest_categorical(name, choices)
        for name, choices in optuna_search_grid.items()
    }
    return tie_lstm_params(raw_params)

def objective(trial: optuna.Trial) -> float:
    params = suggest_lstm_params(trial)
    start_time = time.time()
    row = {"trial_number": trial.number, **params}
    model = LSTM(
        h=H,
        input_size=int(params["input_size"]),
        futr_exog_list=futr_exog_cols,
        loss=MAE(),
        max_steps=int(params["max_steps"]),
        scaler_type=params["scaler_type"],
        encoder_hidden_size=int(params["encoder_hidden_size"]),
        decoder_hidden_size=int(params["decoder_hidden_size"]),
        encoder_n_layers=int(params["encoder_n_layers"]),
        decoder_layers=int(params["decoder_layers"]),
        batch_size=int(params["batch_size"]),
        learning_rate=float(params["learning_rate"]),
        random_seed=int(params["random_seed"]),
        alias=PRED_COL,
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )

    nf_model = NeuralForecast(models=[model], freq="D")
    cv_result = nf_model.cross_validation(
        df=train_val_df,
        n_windows=N_WINDOWS,
        step_size=H,
        refit=False,
        verbose=False,
    )

    cv_result = cv_result.sort_values(["unique_id", "cutoff", "ds"]).reset_index(drop=True)

    validation_wape = forecast_wape(cv_result, pred_col=PRED_COL)
    row.update(
        {
            "training_minutes": (time.time() - start_time) / 60,
            "forecast_rows": len(cv_result),
            "validation_windows": cv_result["cutoff"].nunique(),
            "WAPE": validation_wape,
        }
    )
    objective_value = float(validation_wape) if np.isfinite(validation_wape) else float("inf")
    for key, value in row.items():
        trial.set_user_attr(key, value)
    validation_results.append(row)
    return objective_value


study.optimize(objective, n_trials=optuna_n_trials, gc_after_trial=True)
validation_results_df = (
    pd.DataFrame(validation_results)
    .sort_values("WAPE", na_position="last")
    .reset_index(drop=True)
)

Parenkami geriausi parametrai pagal validacijos WAPE.


In [ ]:
valid_wape_results = validation_results_df[validation_results_df["WAPE"].notna()].copy()

tied_result_mask = (
    validation_results_df["encoder_hidden_size"].astype(int).eq(validation_results_df["decoder_hidden_size"].astype(int))
    & validation_results_df["encoder_n_layers"].astype(int).eq(validation_results_df["decoder_layers"].astype(int))
)
valid_wape_results = valid_wape_results.loc[tied_result_mask.loc[valid_wape_results.index]].copy()

best_params = valid_wape_results.iloc[0].to_dict()
best_input_size = int(best_params["input_size"])
best_encoder_hidden_size = int(best_params["encoder_hidden_size"])
best_decoder_hidden_size = int(best_params["decoder_hidden_size"])
best_encoder_n_layers = int(best_params["encoder_n_layers"])
best_decoder_layers = int(best_params["decoder_layers"])
best_max_steps = int(best_params["max_steps"])
best_batch_size = int(best_params["batch_size"])
best_learning_rate = float(best_params["learning_rate"])
best_scaler_type = str(best_params["scaler_type"])
best_random_seed = int(best_params["random_seed"])

Su geriausiais parametrais gaunamos LSTM prognozės testavimo aibei.


In [ ]:
best_model = LSTM(
    h=H,
    input_size=best_input_size,
    futr_exog_list=futr_exog_cols,
    loss=MAE(),
    max_steps=best_max_steps,
    scaler_type=best_scaler_type,
    encoder_hidden_size=best_encoder_hidden_size,
    decoder_hidden_size=best_decoder_hidden_size,
    encoder_n_layers=best_encoder_n_layers,
    decoder_layers=best_decoder_layers,
    batch_size=best_batch_size,
    learning_rate=best_learning_rate,
    random_seed=best_random_seed,
    alias=PRED_COL,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=False,
    enable_model_summary=False,
)

nf_best = NeuralForecast(models=[best_model], freq="D")

test_cv_df = nf_best.cross_validation(
    df=Y_df_full,
    n_windows=N_WINDOWS,
    step_size=H,
    refit=False,
    verbose=False,
)

test_cv_df = test_cv_df.sort_values(["unique_id", "cutoff", "ds"]).reset_index(drop=True)

Atrenkamos 46 laiko eilutės.


In [ ]:
test_cv_df_46 = test_cv_df[test_cv_df["unique_id"].isin(selected_46_ids)].copy()
test_cv_df_46 = test_cv_df_46.merge(selected_metadata, on="unique_id", how="left")
test_predictions_46 = test_cv_df_46.copy()

Išsaugomos galutinės LSTM prognozės.


In [ ]:
test_predictions_46.to_parquet(TEST_PREDICTIONS_46_FILE, index=False)